# Neural G2P Model with Perplexity Measurement

Trains a **sequence-to-sequence grapheme-to-phoneme (G2P) model** using the `ipa-mappings` package as the phoneme target alphabet and label source.

### What this does
1. **Data pipeline**: For a chosen language, uses `PhonetokTokenizer.ipa_beam()` to generate `(grapheme_sequence, IPA_sequence)` training pairs from a word list
2. **Model**: A lightweight Transformer encoder-decoder (character-level input, IPA token output)
3. **Perplexity**: Measured on held-out words — how surprised is the model by the correct phoneme sequence? Lower = better G2P model
4. **Cross-lingual perplexity**: Train on one language, evaluate on a related language — measures how much phonological knowledge transfers across the ancestry graph
5. **Export**: ONNX export for low-latency inference

### Key insight for this project
Perplexity as a metric tells you which grapheme→IPA mappings are **well-conditioned** (low perplexity, deterministic) vs. **ambiguous** (high perplexity, like English ⟨ough⟩). This helps identify which entries in the package's grapheme maps need better disambiguation or context rules.

In [ ]:
# ── Dependencies ──────────────────────────────────────────────────────────────
%pip install torch==2.2.0 --index-url https://download.pytorch.org/whl/cpu  # PyTorch (CPU; change to cu121 for GPU)
%pip install numpy==1.26.4           # array ops
%pip install datasets==2.19.0        # HF word-list datasets
%pip install huggingface_hub==0.23.0 # upload gate
%pip install onnx==1.16.0            # ONNX graph utilities
%pip install onnxruntime==1.17.3     # ONNX CPU inference
%pip install mlflow==2.12.2          # experiment tracking
%pip install matplotlib==3.8.4       # loss curves
%pip install seaborn==0.13.2         # plots
%pip install pandas==2.2.2           # results tables
%pip install tqdm==4.66.2            # progress bars
%pip install psutil==5.9.8           # memory monitoring
# %pip install ipa-mappings           # uncomment if not using local path

In [ ]:
import os, gc, math
import psutil

# ── Configuration ─────────────────────────────────────────────────────────────

# Language to train on (primary)
LANG_CODE             = os.getenv("LANG_CODE", "es")             # main training language
EVAL_LANG_CODES       = os.getenv("EVAL_LANG_CODES", "pt,it,fr") # langs for cross-lingual perplexity

# Word list source: "hf" (HuggingFace dataset) | "file" (local .txt) | "builtin" (grapheme keys)
WORD_SOURCE           = os.getenv("WORD_SOURCE", "builtin")      # start with builtin, upgrade to hf
DATASET_REPO_ID       = os.getenv("DATASET_REPO_ID", "cc100")    # HF dataset with text (for word extraction)
DATASET_CONFIG        = os.getenv("DATASET_CONFIG", "es")        # language subset
DATASET_MAX_SAMPLES   = int(os.getenv("DATASET_MAX_SAMPLES", "50000"))  # cap corpus size
STREAM_THRESHOLD_GB   = float(os.getenv("STREAM_THRESHOLD_GB", "2.0"))
WORD_LIST_PATH        = os.getenv("WORD_LIST_PATH", "")          # path to local word list .txt

# G2P tokenizer
BEAM_WIDTH            = int(os.getenv("BEAM_WIDTH", "1"))         # beam width for IPA label generation
MAX_WORD_LEN          = int(os.getenv("MAX_WORD_LEN", "30"))      # discard longer words
MIN_WORD_LEN          = int(os.getenv("MIN_WORD_LEN", "3"))

# Model architecture
D_MODEL               = int(os.getenv("D_MODEL", "128"))          # transformer hidden dim
N_HEADS               = int(os.getenv("N_HEADS", "4"))            # attention heads
N_ENC_LAYERS          = int(os.getenv("N_ENC_LAYERS", "3"))       # encoder layers
N_DEC_LAYERS          = int(os.getenv("N_DEC_LAYERS", "3"))       # decoder layers
FFN_DIM               = int(os.getenv("FFN_DIM", "256"))          # feedforward dim
DROPOUT               = float(os.getenv("DROPOUT", "0.1"))

# Training
TRAIN_EPOCHS          = int(os.getenv("TRAIN_EPOCHS", "30"))
TRAIN_BATCH_SIZE      = int(os.getenv("TRAIN_BATCH_SIZE", "64"))
TRAIN_LR              = float(os.getenv("TRAIN_LR", "3e-4"))
TRAIN_WARMUP_STEPS    = int(os.getenv("TRAIN_WARMUP_STEPS", "200"))
TRAIN_DEVICE          = os.getenv("TRAIN_DEVICE", "cuda" if __import__('torch').cuda.is_available() else "cpu")
TRAIN_SEED            = int(os.getenv("TRAIN_SEED", "42"))
TRAIN_SPLIT           = float(os.getenv("TRAIN_SPLIT", "0.85"))   # train/val split ratio

# Output
OUTPUT_DIR            = os.getenv("OUTPUT_DIR", "./outputs")
ONNX_EXPORT_PATH      = os.getenv("ONNX_EXPORT_PATH", f"{OUTPUT_DIR}/g2p_{LANG_CODE}.onnx")
ONNX_OPSET            = int(os.getenv("ONNX_OPSET", "17"))
QUANTIZE_MODE         = os.getenv("QUANTIZE_MODE", "dynamic")
QUANTIZE_OUTPUT_PATH  = os.getenv("QUANTIZE_OUTPUT_PATH", f"{OUTPUT_DIR}/g2p_{LANG_CODE}_int8.onnx")

# MLflow
MLFLOW_TRACKING_URI   = os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5000")
MLFLOW_EXPERIMENT     = os.getenv("MLFLOW_EXPERIMENT", "g2p-perplexity")
MLFLOW_RUN_NAME       = os.getenv("MLFLOW_RUN_NAME", None)

# HuggingFace
HF_TOKEN              = os.getenv("HF_TOKEN", None)
HF_MODEL_REPO         = os.getenv("HF_MODEL_REPO", None)
HF_UPLOAD_MODEL       = os.getenv("HF_UPLOAD_MODEL", "false").lower() == "true"

os.makedirs(OUTPUT_DIR, exist_ok=True)

import torch
torch.manual_seed(TRAIN_SEED)

def memory_checkpoint(label):
    gc.collect()
    m = psutil.virtual_memory()
    print(f"🧠 [{label}] RAM: {m.used/1e9:.2f} GB / {m.total/1e9:.2f} GB ({m.percent:.1f}%)")

print(f"✅ Configuration loaded | lang={LANG_CODE} | device={TRAIN_DEVICE}")
print(f"   Model: d={D_MODEL}, heads={N_HEADS}, enc={N_ENC_LAYERS}, dec={N_DEC_LAYERS}")

In [ ]:
# ── MLflow setup ──────────────────────────────────────────────────────────────
try:
    import mlflow
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    mlflow.set_experiment(MLFLOW_EXPERIMENT)
    MLFLOW_AVAILABLE = True
    mlflow.start_run(run_name=MLFLOW_RUN_NAME)
    mlflow.log_params({
        "lang": LANG_CODE, "d_model": D_MODEL, "n_heads": N_HEADS,
        "n_enc": N_ENC_LAYERS, "n_dec": N_DEC_LAYERS, "epochs": TRAIN_EPOCHS,
        "lr": TRAIN_LR, "batch": TRAIN_BATCH_SIZE, "beam": BEAM_WIDTH,
    })
    print("✅ MLflow run started")
except Exception as e:
    MLFLOW_AVAILABLE = False
    print(f"⚠️  MLflow unavailable: {e}")

def log_metric(key, value, step=None):
    if MLFLOW_AVAILABLE: mlflow.log_metric(key, value, step=step)

def log_artifact(path):
    if MLFLOW_AVAILABLE and os.path.exists(path): mlflow.log_artifact(path)

## 📊 1. Build Training Data from IPA Mappings

In [ ]:
import sys, re
sys.path.insert(0, "..")

import orthography2ipa
from orthography2ipa.phonetok import PhonetokTokenizer

spec = orthography2ipa.get(LANG_CODE)
tok  = PhonetokTokenizer(spec)

print(f"✅ Loaded spec: {spec.name} | {len(spec.graphemes)} grapheme entries")

# ── Build word list ────────────────────────────────────────────────────────────
def clean_word(w):
    w = w.strip().lower()
    if not (MIN_WORD_LEN <= len(w) <= MAX_WORD_LEN):
        return None
    if not re.match(r'^[a-záéíóúüñçàèìòùâêîôûäëïöüãõœæøðþ]+$', w):
        return None
    return w

raw_words = set()

if WORD_SOURCE == "builtin":
    # Use grapheme keys as seed words (good for small experiments)
    # Better: load a real wordlist below
    print("⚠️  Using builtin grapheme keys — WORD_SOURCE='hf' recommended for production")
    # Generate synthetic words by combining grapheme patterns
    grapheme_keys = list(spec.graphemes.keys())
    import itertools, random
    rng = random.Random(TRAIN_SEED)
    for _ in range(20000):
        length = rng.randint(3, 8)
        word = "".join(rng.choice(grapheme_keys) for _ in range(length))
        w = clean_word(word)
        if w: raw_words.add(w)

elif WORD_SOURCE == "file":
    with open(WORD_LIST_PATH) as f:
        for line in f:
            w = clean_word(line.split()[0] if line.split() else "")
            if w: raw_words.add(w)

elif WORD_SOURCE == "hf":
    from datasets import load_dataset
    from huggingface_hub import dataset_info as hf_dataset_info
    import psutil

    try:
        info = hf_dataset_info(DATASET_REPO_ID)
        size_gb = getattr(info, "size_on_disk", 0) / 1e9
    except Exception:
        size_gb = 0.0
    USE_STREAMING = size_gb > STREAM_THRESHOLD_GB
    print(f"📦 Dataset {DATASET_REPO_ID} | {size_gb:.1f} GB | streaming={USE_STREAMING}")

    ds = load_dataset(DATASET_REPO_ID, DATASET_CONFIG, split="train",
                      streaming=USE_STREAMING, token=HF_TOKEN)
    n = 0
    for ex in ds:
        text = ex.get("text", "")
        for word in text.split():
            w = clean_word(word)
            if w: raw_words.add(w)
        n += 1
        if n >= DATASET_MAX_SAMPLES:
            break
    del ds

print(f"   Raw word candidates: {len(raw_words)}")

# ── Generate G2P pairs using PhonetokTokenizer ────────────────────────────────
g2p_pairs = []  # list of (chars_list, ipa_list)
failed = 0
for word in raw_words:
    try:
        paths = tok.ipa_beam(word, beam_width=BEAM_WIDTH)
        if not paths:
            failed += 1
            continue
        ipa_str = paths[0].ipa  # best beam path
        if not ipa_str:
            continue
        g2p_pairs.append((list(word), list(ipa_str)))
    except Exception:
        failed += 1

print(f"✅ G2P pairs: {len(g2p_pairs)} | Failed tokenizations: {failed}")
memory_checkpoint("after data generation")

## 🔍 2. EDA — G2P Pair Statistics

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

grapheme_lens = [len(g) for g, _ in g2p_pairs]
ipa_lens      = [len(p) for _, p in g2p_pairs]
compression   = [len(p) / len(g) for g, p in g2p_pairs]

# IPA token frequency
all_ipa_tokens = [t for _, p in g2p_pairs for t in p]
ipa_freq = Counter(all_ipa_tokens)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(grapheme_lens, bins=range(MIN_WORD_LEN, MAX_WORD_LEN+2), color="steelblue", edgecolor="white")
axes[0].set_title("Word length distribution"); axes[0].set_xlabel("Characters")

axes[1].hist(compression, bins=30, color="darkorange", edgecolor="white")
axes[1].axvline(1.0, color="red", linestyle="--", label="1:1 ratio")
axes[1].set_title("IPA/grapheme length ratio"); axes[1].set_xlabel("ratio"); axes[1].legend()

top_ipa = ipa_freq.most_common(25)
axes[2].barh([t for t, _ in top_ipa][::-1], [c for _, c in top_ipa][::-1], color="seagreen")
axes[2].set_title(f"Top-25 IPA tokens in {spec.name}"); axes[2].set_xlabel("Count")

plt.suptitle(f"G2P Pair Statistics — {spec.name} ({len(g2p_pairs)} pairs)", fontweight="bold")
plt.tight_layout()
eda_path = f"{OUTPUT_DIR}/eda_g2p_{LANG_CODE}.png"
plt.savefig(eda_path, dpi=120, bbox_inches="tight")
log_artifact(eda_path)
plt.show()

unique_ipa_tokens = len(ipa_freq)
print(f"Unique IPA tokens in corpus: {unique_ipa_tokens}")
print(f"Avg grapheme len: {np.mean(grapheme_lens):.1f} | Avg IPA len: {np.mean(ipa_lens):.1f}")
log_metric("n_pairs", len(g2p_pairs))
log_metric("unique_ipa_tokens", unique_ipa_tokens)

In [ ]:
# ── Vocabulary ────────────────────────────────────────────────────────────────
PAD, BOS, EOS, UNK = "<pad>", "<bos>", "<eos>", "<unk>"

# Grapheme vocab: all unique characters seen in corpus
all_chars = sorted(set(c for word, _ in g2p_pairs for c in word))
src_vocab  = [PAD, BOS, EOS, UNK] + all_chars
src_stoi   = {t: i for i, t in enumerate(src_vocab)}

# IPA vocab: all unique IPA tokens (single chars/diacritics)
all_ipa_types = sorted(ipa_freq.keys())
tgt_vocab  = [PAD, BOS, EOS, UNK] + all_ipa_types
tgt_stoi   = {t: i for i, t in enumerate(tgt_vocab)}

SRC_PAD_IDX = src_stoi[PAD]
TGT_PAD_IDX = tgt_stoi[PAD]
TGT_BOS_IDX = tgt_stoi[BOS]
TGT_EOS_IDX = tgt_stoi[EOS]

print(f"Source vocab size: {len(src_vocab)} | Target vocab size: {len(tgt_vocab)}")
log_metric("src_vocab_size", len(src_vocab))
log_metric("tgt_vocab_size", len(tgt_vocab))

def encode_src(chars):
    return [src_stoi.get(c, src_stoi[UNK]) for c in chars]

def encode_tgt(ipa_tokens):
    return [TGT_BOS_IDX] + [tgt_stoi.get(t, tgt_stoi[UNK]) for t in ipa_tokens] + [TGT_EOS_IDX]

In [ ]:
# ── Dataset / DataLoader ──────────────────────────────────────────────────────
import random
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

random.shuffle(g2p_pairs)
split = int(len(g2p_pairs) * TRAIN_SPLIT)
train_pairs = g2p_pairs[:split]
val_pairs   = g2p_pairs[split:]
print(f"Train: {len(train_pairs)} | Val: {len(val_pairs)}")

class G2PDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, i):
        src, tgt = self.pairs[i]
        return torch.tensor(encode_src(src), dtype=torch.long), \
               torch.tensor(encode_tgt(tgt), dtype=torch.long)

def collate_fn(batch):
    srcs, tgts = zip(*batch)
    src_padded = torch.nn.utils.rnn.pad_sequence(srcs, batch_first=True, padding_value=SRC_PAD_IDX)
    tgt_padded = torch.nn.utils.rnn.pad_sequence(tgts, batch_first=True, padding_value=TGT_PAD_IDX)
    return src_padded, tgt_padded

train_dl = DataLoader(G2PDataset(train_pairs), batch_size=TRAIN_BATCH_SIZE, shuffle=True,  collate_fn=collate_fn)
val_dl   = DataLoader(G2PDataset(val_pairs),   batch_size=TRAIN_BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

## 🏋️ 3. Model Definition & Training

In [ ]:
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])


class G2PTransformer(nn.Module):
    """
    Character-level grapheme-to-phoneme seq2seq Transformer.
    Input:  (batch, src_len) grapheme indices
    Output: (batch, tgt_len, tgt_vocab_size) logits
    """
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, n_heads,
                 n_enc_layers, n_dec_layers, ffn_dim, dropout, src_pad, tgt_pad):
        super().__init__()
        self.src_embed = nn.Embedding(src_vocab_size, d_model, padding_idx=src_pad)
        self.tgt_embed = nn.Embedding(tgt_vocab_size, d_model, padding_idx=tgt_pad)
        self.pos_enc   = PositionalEncoding(d_model, dropout=dropout)
        self.transformer = nn.Transformer(
            d_model=d_model, nhead=n_heads,
            num_encoder_layers=n_enc_layers,
            num_decoder_layers=n_dec_layers,
            dim_feedforward=ffn_dim,
            dropout=dropout, batch_first=True,
        )
        self.proj = nn.Linear(d_model, tgt_vocab_size)
        self.src_pad = src_pad
        self.tgt_pad = tgt_pad
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def make_causal_mask(self, sz, device):
        return torch.triu(torch.ones(sz, sz, device=device), diagonal=1).bool()

    def forward(self, src, tgt):
        src_key_padding = (src == self.src_pad)
        tgt_key_padding = (tgt == self.tgt_pad)
        tgt_mask = self.make_causal_mask(tgt.size(1), tgt.device)

        src_emb = self.pos_enc(self.src_embed(src))
        tgt_emb = self.pos_enc(self.tgt_embed(tgt))

        out = self.transformer(
            src_emb, tgt_emb,
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_key_padding,
            tgt_key_padding_mask=tgt_key_padding,
            memory_key_padding_mask=src_key_padding,
        )
        return self.proj(out)  # (batch, tgt_len, tgt_vocab)

    @torch.no_grad()
    def greedy_decode(self, src, max_len=64):
        """Greedy decoding for inference."""
        self.eval()
        device = next(self.parameters()).device
        src = src.to(device)
        memory = self.transformer.encoder(
            self.pos_enc(self.src_embed(src)),
            src_key_padding_mask=(src == self.src_pad)
        )
        tgt = torch.tensor([[TGT_BOS_IDX]] * src.size(0), device=device)
        for _ in range(max_len):
            tgt_emb = self.pos_enc(self.tgt_embed(tgt))
            tgt_mask = self.make_causal_mask(tgt.size(1), device)
            dec_out = self.transformer.decoder(tgt_emb, memory, tgt_mask=tgt_mask)
            next_tok = self.proj(dec_out[:, -1]).argmax(-1, keepdim=True)
            tgt = torch.cat([tgt, next_tok], dim=1)
            if (next_tok == TGT_EOS_IDX).all():
                break
        return tgt[:, 1:]  # strip BOS


device = torch.device(TRAIN_DEVICE)
model = G2PTransformer(
    src_vocab_size=len(src_vocab),
    tgt_vocab_size=len(tgt_vocab),
    d_model=D_MODEL, n_heads=N_HEADS,
    n_enc_layers=N_ENC_LAYERS, n_dec_layers=N_DEC_LAYERS,
    ffn_dim=FFN_DIM, dropout=DROPOUT,
    src_pad=SRC_PAD_IDX, tgt_pad=TGT_PAD_IDX,
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"✅ Model: {n_params:,} parameters | device={device}")
log_metric("n_params", n_params)

In [ ]:
from tqdm import tqdm

criterion = nn.CrossEntropyLoss(ignore_index=TGT_PAD_IDX, label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=TRAIN_LR, weight_decay=0.01)

def lr_lambda(step):
    if step < TRAIN_WARMUP_STEPS:
        return step / max(TRAIN_WARMUP_STEPS, 1)
    return max(0.1, 1.0 - (step - TRAIN_WARMUP_STEPS) / (TRAIN_EPOCHS * len(train_dl)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

def compute_loss_and_ppl(loader):
    """Compute cross-entropy loss and perplexity on a DataLoader."""
    model.eval()
    total_loss, total_tokens = 0.0, 0
    with torch.no_grad():
        for src, tgt in loader:
            src, tgt = src.to(device), tgt.to(device)
            tgt_in  = tgt[:, :-1]
            tgt_out = tgt[:, 1:]
            logits = model(src, tgt_in)
            B, T, V = logits.shape
            loss = criterion(logits.reshape(B * T, V), tgt_out.reshape(B * T))
            # Count non-pad tokens for proper averaging
            mask = (tgt_out != TGT_PAD_IDX)
            n_tok = mask.sum().item()
            raw_loss = F.cross_entropy(
                logits.reshape(B * T, V), tgt_out.reshape(B * T),
                ignore_index=TGT_PAD_IDX, reduction="sum"
            )
            total_loss   += raw_loss.item()
            total_tokens += n_tok
    avg_loss = total_loss / max(total_tokens, 1)
    ppl = math.exp(min(avg_loss, 100))  # clamp to avoid overflow
    return avg_loss, ppl

train_losses, val_losses, val_ppls = [], [], []
best_val_ppl = float("inf")
global_step = 0

print(f"Training {TRAIN_EPOCHS} epochs on {len(train_pairs)} pairs...")
for epoch in range(1, TRAIN_EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    for src, tgt in tqdm(train_dl, desc=f"Epoch {epoch}/{TRAIN_EPOCHS}", leave=False):
        src, tgt = src.to(device), tgt.to(device)
        tgt_in  = tgt[:, :-1]
        tgt_out = tgt[:, 1:]
        optimizer.zero_grad()
        logits = model(src, tgt_in)
        B, T, V = logits.shape
        loss = criterion(logits.reshape(B * T, V), tgt_out.reshape(B * T))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        epoch_loss += loss.item()
        global_step += 1

    avg_train_loss = epoch_loss / len(train_dl)
    val_loss, val_ppl = compute_loss_and_ppl(val_dl)
    train_losses.append(avg_train_loss)
    val_losses.append(val_loss)
    val_ppls.append(val_ppl)

    log_metric("train_loss", avg_train_loss, step=epoch)
    log_metric("val_loss",   val_loss,       step=epoch)
    log_metric("val_ppl",    val_ppl,        step=epoch)

    if val_ppl < best_val_ppl:
        best_val_ppl = val_ppl
        torch.save(model.state_dict(), f"{OUTPUT_DIR}/best_model_{LANG_CODE}.pt")

    if epoch % 5 == 0 or epoch == TRAIN_EPOCHS:
        print(f"Epoch {epoch:3d} | train_loss={avg_train_loss:.4f} | val_loss={val_loss:.4f} | val_ppl={val_ppl:.2f}")

# Load best model
model.load_state_dict(torch.load(f"{OUTPUT_DIR}/best_model_{LANG_CODE}.pt", map_location=device))
print(f"\n✅ Training complete | Best val perplexity: {best_val_ppl:.2f}")
log_metric("best_val_ppl", best_val_ppl)
memory_checkpoint("after training")

## 📈 4. Metrics, Loss Curves & Perplexity Analysis

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
epochs_x = list(range(1, TRAIN_EPOCHS + 1))
axes[0].plot(epochs_x, train_losses, label="Train loss", color="steelblue")
axes[0].plot(epochs_x, val_losses,   label="Val loss",   color="darkorange")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Cross-entropy loss")
axes[0].set_title(f"G2P Training — {spec.name}")
axes[0].legend()

# Perplexity curve
axes[1].plot(epochs_x, val_ppls, color="seagreen", marker="o", markersize=3)
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Perplexity (PPL)")
axes[1].set_title("Validation Perplexity")
axes[1].axhline(best_val_ppl, color="red", linestyle="--", alpha=0.5, label=f"Best={best_val_ppl:.2f}")
axes[1].legend()

plt.tight_layout()
curves_path = f"{OUTPUT_DIR}/training_curves_{LANG_CODE}.png"
plt.savefig(curves_path, dpi=120, bbox_inches="tight")
log_artifact(curves_path)
plt.show()

In [ ]:
# ── Per-grapheme perplexity: which graphemes confuse the model most? ──────────
# Evaluate perplexity of individual grapheme positions

model.eval()
grapheme_ppl = {}  # grapheme char -> list of per-token losses

with torch.no_grad():
    for src_chars, ipa_tokens in val_pairs[:2000]:
        src_t = torch.tensor(encode_src(src_chars), dtype=torch.long).unsqueeze(0).to(device)
        tgt_t = torch.tensor(encode_tgt(ipa_tokens), dtype=torch.long).unsqueeze(0).to(device)
        logits = model(src_t, tgt_t[:, :-1])  # (1, T, V)
        tgt_out = tgt_t[:, 1:].squeeze(0)
        token_losses = F.cross_entropy(
            logits.squeeze(0), tgt_out, reduction="none"
        ).cpu().numpy()  # (T,)
        # Attribute loss of each IPA position back to the source grapheme
        # by simple proportional mapping
        n_src, n_ipa = len(src_chars), len(ipa_tokens)
        for j, loss_j in enumerate(token_losses[:n_ipa]):
            src_idx = min(int(j * n_src / max(n_ipa, 1)), n_src - 1)
            g = src_chars[src_idx]
            grapheme_ppl.setdefault(g, []).append(float(loss_j))

grapheme_ppl_mean = {g: math.exp(min(np.mean(v), 10)) for g, v in grapheme_ppl.items()}
gp_sorted = sorted(grapheme_ppl_mean.items(), key=lambda x: -x[1])

print(f"\nTop-15 most ambiguous graphemes in {spec.name} (highest perplexity):")
for g, ppl_val in gp_sorted[:15]:
    ipa_options = spec.graphemes.get(g, [])
    print(f"  '{g}' → {ipa_options}   ppl={ppl_val:.3f}")

# Plot
top_g = gp_sorted[:20]
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh([g for g, _ in top_g][::-1], [p for _, p in top_g][::-1],
        color=["tomato" if p > 2.0 else "steelblue" for _, p in top_g[::-1]])
ax.set_xlabel("Mean perplexity")
ax.set_title(f"Grapheme-Level Perplexity — {spec.name}\n(red = high ambiguity)")
plt.tight_layout()
gp_path = f"{OUTPUT_DIR}/grapheme_perplexity_{LANG_CODE}.png"
plt.savefig(gp_path, dpi=120, bbox_inches="tight")
log_artifact(gp_path)
plt.show()

In [ ]:
# ── Cross-lingual perplexity ───────────────────────────────────────────────────
# Evaluate the trained model (trained on LANG_CODE) on related languages.
# Phonologically similar languages should have lower cross-lingual PPL.

eval_langs = [c.strip() for c in EVAL_LANG_CODES.split(",") if c.strip()]
crosslingual_results = {LANG_CODE: best_val_ppl}  # baseline: same-language ppl

for eval_code in eval_langs:
    try:
        eval_spec = orthography2ipa.get(eval_code)
        eval_tok  = PhonetokTokenizer(eval_spec)
    except Exception as e:
        print(f"⚠️  Could not load {eval_code}: {e}")
        continue

    # Generate evaluation pairs using source vocab (shared grapheme space)
    eval_pairs_raw = []
    for word in list(raw_words)[:3000]:
        try:
            paths = eval_tok.ipa_beam(word, beam_width=1)
            if paths and paths[0].ipa:
                eval_pairs_raw.append((list(word), list(paths[0].ipa)))
        except Exception:
            pass

    if len(eval_pairs_raw) < 50:
        print(f"⚠️  Not enough eval pairs for {eval_code} ({len(eval_pairs_raw)})")
        continue

    # Build DataLoader using *this model's vocabulary* (unknown IPA -> UNK)
    eval_dl_raw = DataLoader(
        G2PDataset(eval_pairs_raw),
        batch_size=TRAIN_BATCH_SIZE, shuffle=False, collate_fn=collate_fn
    )
    _, cl_ppl = compute_loss_and_ppl(eval_dl_raw)
    crosslingual_results[eval_code] = cl_ppl
    print(f"  {eval_spec.name:25s} ({eval_code}): cross-lingual PPL = {cl_ppl:.2f}")
    log_metric(f"crosslingual_ppl_{eval_code}", cl_ppl)

# Summary plot
import pandas as pd
cl_df = pd.DataFrame([(k, v) for k, v in crosslingual_results.items()], columns=["lang", "ppl"])
cl_df = cl_df.sort_values("ppl")

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(cl_df["lang"], cl_df["ppl"],
              color=["steelblue" if l == LANG_CODE else "darkorange" for l in cl_df["lang"]])
ax.set_ylabel("Perplexity")
ax.set_title(f"Cross-Lingual Perplexity\n(model trained on {spec.name}, evaluated on related languages)")
ax.axhline(best_val_ppl, linestyle="--", color="red", alpha=0.7, label=f"{LANG_CODE} baseline={best_val_ppl:.1f}")
ax.legend()
plt.tight_layout()
cl_path = f"{OUTPUT_DIR}/crosslingual_ppl_{LANG_CODE}.png"
plt.savefig(cl_path, dpi=120, bbox_inches="tight")
log_artifact(cl_path)
plt.show()
print(cl_df.to_string(index=False))

## ✅ 5. Model Verification — Sample Predictions

In [ ]:
def predict(word: str) -> str:
    """Run greedy G2P prediction for a single word."""
    src_t = torch.tensor(encode_src(list(word)), dtype=torch.long).unsqueeze(0)
    pred_tokens = model.greedy_decode(src_t, max_len=60).squeeze(0)
    tokens = [tgt_vocab[t.item()] for t in pred_tokens
              if t.item() not in (TGT_PAD_IDX, TGT_BOS_IDX, TGT_EOS_IDX)]
    return "".join(tokens)

def ground_truth(word: str) -> str:
    """Get gold IPA from PhonetokTokenizer."""
    paths = tok.ipa_beam(word, beam_width=1)
    return paths[0].ipa if paths else "?"

test_words = [g2p_pairs[i][0] for i in range(0, min(20, len(val_pairs)))]
test_words = ["".join(w) for w in test_words]

print(f"\n{'Word':20s}  {'Predicted IPA':25s}  {'Gold IPA':25s}  {'Match'}")
print("-" * 80)
correct = 0
for word in test_words:
    pred = predict(word)
    gold = ground_truth(word)
    match = pred == gold
    correct += int(match)
    print(f"{word:20s}  {pred:25s}  {gold:25s}  {'✅' if match else '❌'}")

acc = correct / len(test_words)
print(f"\nWord-level accuracy on sample: {acc:.2%}")
log_metric("sample_word_accuracy", acc)

## 📤 6. ONNX Export

In [ ]:
import onnx

# Export encoder-only for embedding extraction (useful standalone)
# Full seq2seq ONNX export requires separate encoder/decoder passes;
# here we export a single forward pass with fixed-length teacher-forcing
# for use as a scoring/perplexity oracle.

class G2PScoringWrapper(nn.Module):
    """Wraps G2PTransformer for ONNX: (src, tgt_in) -> logits."""
    def __init__(self, inner): super().__init__(); self.inner = inner
    def forward(self, src, tgt_in): return self.inner(src, tgt_in)

wrapper = G2PScoringWrapper(model).cpu()
wrapper.eval()

# Dummy inputs — batch=1, src_len=10, tgt_len=8
dummy_src = torch.randint(1, len(src_vocab), (1, 10), dtype=torch.long)
dummy_tgt = torch.randint(1, len(tgt_vocab), (1, 8),  dtype=torch.long)
dummy_tgt[0, 0] = TGT_BOS_IDX

with torch.no_grad():
    ref_out = wrapper(dummy_src, dummy_tgt)
print(f"Reference output shape: {ref_out.shape}")

torch.onnx.export(
    wrapper,
    (dummy_src, dummy_tgt),
    ONNX_EXPORT_PATH,
    opset_version=ONNX_OPSET,
    input_names=["src", "tgt_in"],
    output_names=["logits"],
    dynamic_axes={
        "src":    {0: "batch", 1: "src_len"},
        "tgt_in": {0: "batch", 1: "tgt_len"},
        "logits": {0: "batch", 1: "tgt_len"},
    },
)

onnx_model = onnx.load(ONNX_EXPORT_PATH)
onnx.checker.check_model(onnx_model)
size_mb = os.path.getsize(ONNX_EXPORT_PATH) / 1e6
print(f"✅ ONNX exported | {size_mb:.1f} MB → {ONNX_EXPORT_PATH}")
log_artifact(ONNX_EXPORT_PATH)
log_metric("onnx_size_mb", size_mb)

In [ ]:
# ── Quantization ──────────────────────────────────────────────────────────────
if QUANTIZE_MODE != "none":
    from onnxruntime.quantization import quantize_dynamic, QuantType
    quantize_dynamic(
        model_input=ONNX_EXPORT_PATH,
        model_output=QUANTIZE_OUTPUT_PATH,
        weight_type=QuantType.QInt8,
    )
    q_size_mb = os.path.getsize(QUANTIZE_OUTPUT_PATH) / 1e6
    compression = (1 - q_size_mb / size_mb) * 100
    print(f"✅ Quantized (int8): {q_size_mb:.1f} MB | Compression: {compression:.1f}%")
    log_metric("quantized_size_mb", q_size_mb)
    log_metric("compression_pct", compression)
else:
    print("⏭️  Quantization skipped")

## ⚡ ONNX Inference Benchmark
*(onnxruntime + numpy only below this point)*

In [ ]:
import onnxruntime as ort
import numpy as np
import timeit, json

inf_path = QUANTIZE_OUTPUT_PATH if (QUANTIZE_MODE != "none" and os.path.exists(QUANTIZE_OUTPUT_PATH)) else ONNX_EXPORT_PATH
sess_opts = ort.SessionOptions()
sess_opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
sess = ort.InferenceSession(inf_path, sess_opts)
print(f"✅ ONNX session: {inf_path}")
print(f"   Inputs:  {[i.name for i in sess.get_inputs()]}")
print(f"   Outputs: {[o.name for o in sess.get_outputs()]}")

# Verify ONNX vs PyTorch
src_np  = dummy_src.numpy()
tgt_np  = dummy_tgt.numpy()
onnx_out = sess.run(["logits"], {"src": src_np, "tgt_in": tgt_np})[0]
with torch.no_grad():
    pt_out = wrapper(dummy_src, dummy_tgt).numpy()
match = np.allclose(pt_out, onnx_out, atol=1e-3)
print(f"PyTorch vs ONNX: {'✅' if match else '⚠️'} | max diff: {np.abs(pt_out - onnx_out).max():.2e}")

# Benchmark
BENCH_SAMPLES, BENCH_WARMUP = 200, 10
for _ in range(BENCH_WARMUP):
    sess.run(["logits"], {"src": src_np, "tgt_in": tgt_np})

latencies = []
for _ in range(BENCH_SAMPLES):
    t0 = timeit.default_timer()
    sess.run(["logits"], {"src": src_np, "tgt_in": tgt_np})
    latencies.append((timeit.default_timer() - t0) * 1000)

lat = np.array(latencies)
bench = {
    "mean_ms": float(np.mean(lat)),
    "p50_ms":  float(np.percentile(lat, 50)),
    "p95_ms":  float(np.percentile(lat, 95)),
    "p99_ms":  float(np.percentile(lat, 99)),
    "throughput_sps": float(1000.0 / np.mean(lat)),
}

print(f"\n⚡ Benchmark (src_len=10, tgt_len=8):")
for k, v in bench.items():
    print(f"   {k}: {v:.3f}")

bench_path = f"{OUTPUT_DIR}/benchmark_{LANG_CODE}.json"
with open(bench_path, "w") as f:
    json.dump(bench, f, indent=2)
log_artifact(bench_path)
for k, v in bench.items():
    if isinstance(v, float): log_metric(f"onnx_{k}", v)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(lat, bins=30, color="steelblue", edgecolor="white")
ax.axvline(bench["p50_ms"], color="orange", linestyle="--", label=f"P50 {bench['p50_ms']:.2f}ms")
ax.axvline(bench["p95_ms"], color="red",    linestyle="--", label=f"P95 {bench['p95_ms']:.2f}ms")
ax.set_xlabel("Latency (ms)"); ax.set_ylabel("Count")
ax.set_title("ONNX G2P Inference Latency"); ax.legend()
plt.tight_layout()
bplot_path = f"{OUTPUT_DIR}/benchmark_latency_{LANG_CODE}.png"
plt.savefig(bplot_path, dpi=120, bbox_inches="tight")
log_artifact(bplot_path)
plt.show()

## ☁️ 7. Upload Gates

In [ ]:
# ── Save vocab files for inference ────────────────────────────────────────────
import json as _json
with open(f"{OUTPUT_DIR}/src_vocab_{LANG_CODE}.json", "w") as f:
    _json.dump(src_vocab, f, ensure_ascii=False)
with open(f"{OUTPUT_DIR}/tgt_vocab_{LANG_CODE}.json", "w") as f:
    _json.dump(tgt_vocab, f, ensure_ascii=False)
print(f"✅ Vocab files saved to {OUTPUT_DIR}")

# ── Upload to HuggingFace ─────────────────────────────────────────────────────
if HF_UPLOAD_MODEL and HF_TOKEN and HF_MODEL_REPO:
    from huggingface_hub import HfApi, login
    login(token=HF_TOKEN)
    api = HfApi(token=HF_TOKEN)
    for fpath in [
        ONNX_EXPORT_PATH, QUANTIZE_OUTPUT_PATH,
        f"{OUTPUT_DIR}/src_vocab_{LANG_CODE}.json",
        f"{OUTPUT_DIR}/tgt_vocab_{LANG_CODE}.json",
    ]:
        if os.path.exists(fpath):
            api.upload_file(
                path_or_fileobj=fpath,
                path_in_repo=os.path.basename(fpath),
                repo_id=HF_MODEL_REPO, repo_type="model"
            )
    print(f"✅ Uploaded to https://huggingface.co/{HF_MODEL_REPO}")
else:
    print("⏭️  HF upload skipped")

if MLFLOW_AVAILABLE:
    mlflow.end_run()
    print("✅ MLflow run ended")